In [ ]:
import io
import threading
import time
import traceback

import numpy as np
import ipywidgets as w
from IPython.display import display
from matplotlib.figure import Figure
from pynq import Overlay, MMIO
from pynq.lib.iic import AxiIIC

ol = Overlay("bits/goertzel.bit")
gz_ip = ol.ip_dict["goertzel_up_wrapper_0"]
i2s_ip = ol.ip_dict["i2s_receiver_0"]
gz = MMIO(gz_ip["phys_addr"], gz_ip["addr_range"])
i2s = MMIO(i2s_ip["phys_addr"], i2s_ip["addr_range"])
iic = AxiIIC(ol.ip_dict["axi_iic_0"])

GZ_EN, GZ_LEN, GZ_CFG, GZ_CNT, GZ_OVR, GZ_RAW = 0x0C, 0x10, 0x14, 0x18, 0x1C, 0x20
GZ_COEF, GZ_S1, GZ_S2 = 0x40, 0x60, 0x80
I2S_CTRL, I2S_TIMING = 0x08, 0x20
CODEC = 0x1A

MCLK = 12.286e6
SCLK_DIV = 4
FS = MCLK / (SCLK_DIV * 64)
FRAC = 14
N = 1200
SHIFT = 12
CHAN = 0
LINE_VOL = 0x117

AMP_MIN = 300
RATIO = 2
CONFIRM = 2
PLOT_EVERY = 4

ROWS = [697, 770, 852, 941]
COLS = [1209, 1336, 1477]
KEYS = [["1", "2", "3"], ["4", "5", "6"], ["7", "8", "9"], ["*", "0", "#"]]
FREQS = ROWS + COLS
COEF = np.array([int(round(2 * np.cos(2 * np.pi * f / FS) * 2**FRAC)) for f in FREQS])

In [ ]:
db = np.full(len(FREQS), -100.0)
last_key = "-"
plot_lock = threading.Lock()

image = w.Image(format="png")

fig = Figure(figsize=(9, 3), dpi=80)
ax = fig.add_subplot()
line, = ax.plot(FREQS, db, "o-", lw=1.2)
ax.axhline(20 * np.log10(AMP_MIN / 32768), color="r", ls="--", lw=1)
ax.set_xticks(FREQS)
ax.set_ylim(-100, 0)
ax.set_xlabel("Hz")
ax.set_ylabel("dBFS")
ax.grid(True, alpha=0.3)
fig.tight_layout()


def refresh():
    with plot_lock:
        line.set_ydata(db)
        ax.set_title(f"Key {last_key}")
        png = io.BytesIO()
        fig.savefig(png, format="png")
        image.value = png.getvalue()

In [ ]:
status = w.Label(value="Ready")
err_box = w.Textarea(rows=4, disabled=True, layout=w.Layout(width="600px", display="none"))

cand = None
cand_cnt = 0
cur = None


def set_status(text, err=None):
    status.value = text
    err_box.value = err or ""
    err_box.layout.display = "block" if err else "none"

def _codec(reg, val):
    iic.send(CODEC, bytes([(reg << 1) | (val >> 8), val & 0xFF]), 2)
    time.sleep(0.005)

def init_audio():
    gz.write(GZ_EN, 0)
    i2s.write(I2S_CTRL, 0)
    i2s.write(I2S_TIMING, SCLK_DIV)
    i2s.write(I2S_CTRL, 1)
    _codec(0x0F, 0x000)
    _codec(0x06, 0x030)
    _codec(0x00, LINE_VOL)
    _codec(0x01, LINE_VOL)
    _codec(0x04, 0x010)
    _codec(0x05, 0x000)
    _codec(0x07, 0x002)
    _codec(0x08, 0x000)
    _codec(0x09, 0x001)
    _codec(0x06, 0x020)

def s32(x):
    return x - (1 << 32) if x & 0x80000000 else x

def gz_start():
    gz.write(GZ_EN, 0)
    for i, c in enumerate(COEF):
        gz.write(GZ_COEF + 4 * i, int(c) & 0xFFFFFFFF)
    gz.write(GZ_LEN, N)
    gz.write(GZ_CFG, SHIFT | (CHAN << 8))
    gz.write(GZ_EN, 1)

def gz_stop():
    gz.write(GZ_EN, 0)

def gz_read():
    while True:
        cnt = gz.read(GZ_CNT)
        s1 = [s32(gz.read(GZ_S1 + 4 * i)) for i in range(7)]
        s2 = [s32(gz.read(GZ_S2 + 4 * i)) for i in range(7)]
        if gz.read(GZ_CNT) == cnt:
            return cnt, s1, s2

def amplitudes(s1, s2):
    s1 = np.array(s1, dtype=np.float64)
    s2 = np.array(s2, dtype=np.float64)
    p = s1**2 + s2**2 - (COEF / 2**FRAC) * s1 * s2
    return 2 * np.sqrt(np.maximum(p, 0)) / N

def pick(a):
    i = int(np.argmax(a))
    if a[i] < AMP_MIN or np.sort(a)[-2] * RATIO > a[i]:
        return None
    return i

def detect(amp):
    r = pick(amp[:4])
    c = pick(amp[4:])
    if r is None or c is None:
        return None
    return KEYS[r][c]

def update_key(key):
    global cand, cand_cnt, cur, last_key
    if key == cand:
        cand_cnt += 1
    else:
        cand, cand_cnt = key, 1
    if cand_cnt >= CONFIRM and cur != cand:
        cur = cand
        last_key = cur or "-"
        light(cur)
        if cur:
            seq_box.value += cur

def rx_worker():
    global db
    last_cnt = 0
    n = 0
    try:
        while run.value:
            cnt, s1, s2 = gz_read()
            if cnt == last_cnt:
                time.sleep(0.005)
                continue
            last_cnt = cnt
            amp = amplitudes(s1, s2)
            key = detect(amp)
            update_key(key)
            db = 20 * np.log10(amp / 32768 + 1e-6)
            set_status(f"Block {cnt}, key {key or '-'}")
            n += 1
            if n % PLOT_EVERY == 0:
                refresh()
    except Exception:
        set_status("Error: receive stopped", traceback.format_exc())

In [ ]:
def safe(fn, *args):
    try:
        fn(*args)
    except Exception:
        set_status("Error", traceback.format_exc())


def make_key(k):
    return w.Button(description=k, layout=w.Layout(width="60px", height="50px"))


def light(key):
    for k, b in btns.items():
        b.button_style = "success" if k == key else ""


def on_run(c):
    global cand, cand_cnt, cur
    if c["new"]:
        cand, cand_cnt, cur = None, 0, None
        run.description = "Stop"
        safe(gz_start)
        threading.Thread(target=rx_worker, daemon=True).start()
    else:
        run.description = "Start"
        safe(gz_stop)
        light(None)
        set_status("Stopped")


def on_clear(_):
    seq_box.value = ""


btns = {k: make_key(k) for row in KEYS for k in row}
pad = w.GridBox(list(btns.values()), layout=w.Layout(grid_template_columns="repeat(3, 60px)"))
seq_box = w.Text(value="", description="Sequence", disabled=True, style={"description_width": "110px"}, layout=w.Layout(width="520px"))
run = w.ToggleButton(value=False, description="Start", layout=w.Layout(width="80px"))
clear = w.Button(description="Clear", layout=w.Layout(width="80px"))

run.observe(on_run, "value")
clear.on_click(on_clear)

safe(init_audio)
refresh()

display(w.VBox([
    w.HBox([run, clear]),
    pad,
    seq_box,
    status,
    err_box,
    image,
]))

run.value = True